# Overnight Clean-Slate Pipeline

This notebook is the notebook-first overnight controller for the DGGR reset path.

It is designed to answer the actual research question:

- can we preserve musical identity
- while generating audio that convincingly sounds as if it belongs to the target genre manifold?

## Important framing

When we say the codec branch is a strong editor, that does **not** mean it is already producing dramatic remasters.
It means it is strong at the following narrower job:

- staying stable
- preserving melody and structure
- avoiding complete collapse

The current architecture is still strongly biased toward faithfulness to the source, so the edits often do not branch away far enough.
That is exactly why this overnight workflow runs **both** a fresh codec baseline and a two-phase diffusion path, then judges them with realism plus target-style movement.

## What this notebook does

1. Refreshes the pipeline audit.
2. Selects random monitor and long-form clips from `Downloads/`.
3. Runs a fresh codec training run from scratch.
4. Sweeps the codec checkpoints with the realism supervisor.
5. Runs a fresh diffusion V2 training run from scratch.
6. Sweeps all V2 diffusion checkpoints with the realism supervisor.
7. Fine-tunes a diffusion V3 run from the best V2 checkpoint.
8. Sweeps all V3 checkpoints with the realism supervisor.
9. Selects the best diffusion candidate across V2 and V3, then runs long-form generation on random Downloads clips.
10. Writes a morning summary with the key artifact paths.

This notebook reuses the **validated Lab 1 checkpoint** and the existing feature caches. That is intentional.
The reset here means **fresh model training without inheriting old run weights**, not retraining every preprocessing stage unnecessarily.

In [ ]:
from pathlib import Path
import json
import sys


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'README.md').exists() and (p / 'lab 3.1').exists():
            return p
    raise RuntimeError('Could not locate repo root from notebook cwd.')


REPO = find_repo_root(Path.cwd().resolve())
SCRIPTS = REPO / 'lab 3.1' / 'scripts'
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

import overnight_pipeline as op

REPO

In [ ]:
# Primary control cell.
# Set RUN_ALL = True when you are ready to leave this running overnight.

RUN_ALL = True

cfg = op.OvernightConfig(
    downloads_dir=Path.home() / 'Downloads',
    work_root=REPO / 'lab 3.1' / 'outputs' / 'overnight_runs',
    codec_reuse_cache_dir=REPO / 'saves2' / 'lab3_codec_transfer' / 'run1051' / 'cache',
    diffusion_cache_dir=REPO / 'saves2' / 'lab3_diffusion' / 'run_d001' / 'cache',
    lab1_checkpoint=REPO / 'saves' / 'lab1_run_combo_af_gate_exit_v2' / 'latest.pt',
    seed=328,
    codec_stage1_epochs=8,
    codec_stage2_epochs=10,
    codec_stage3_epochs=6,
    codec_max_batches_per_epoch=96,
    diffusion_epochs=18,
    diffusion_epoch_samples=6,
    longform_seconds=75.0,
    run_audit=True,
    run_codec=True,
    run_codec_sweep=True,
    run_diffusion_v2=True,
    run_diffusion_v2_sweep=True,
    run_diffusion_v3=True,
    run_diffusion_v3_sweep=True,
    run_longform=True,
).materialize()

cfg

In [ ]:
clip_plan = op.prepare_clip_plan(cfg)
clip_plan_path = op.save_clip_plan(cfg, clip_plan)
print('Clip plan saved to:', clip_plan_path)
op.print_clip_plan(clip_plan)

In [ ]:
print('Codec command:')
print(' '.join(str(x) for x in op.build_codec_command(cfg, clip_plan)))
print('\nDiffusion V2 command:')
print(' '.join(str(x) for x in op.build_diffusion_command(cfg)))
print('\nDiffusion V3 command (uses the best V2 checkpoint once the V2 sweep finishes):')
print(' '.join(str(x) for x in op.build_diffusion_v3_command(cfg, v2_checkpoint=REPO / 'PLACEHOLDER_V2_CHECKPOINT.pt')))
print('\nLong-form commands (preview):')
for cmd in op.build_longform_commands(cfg, clip_plan, REPO / 'PLACEHOLDER_CHECKPOINT.pt'):
    print(' '.join(str(x) for x in cmd))
    print()

## Execution

When `RUN_ALL = True`, the cell below will run the full overnight sequence.

Expected outputs by morning:

- refreshed audit tables
- a fresh codec run folder
- codec realism supervisor output
- a fresh diffusion V2 run folder
- diffusion V2 realism supervisor output
- a diffusion V3 fine-tune run folder
- diffusion V3 realism supervisor output
- long-form diffusion outputs on random Downloads clips
- `morning_summary.json` and `morning_summary.md`

All logs are written under:

`lab 3.1/outputs/overnight_runs/<tag>/logs/`

In [ ]:
summary = None
if RUN_ALL:
    summary = op.run_overnight_suite(cfg)
    print('\nOvernight suite finished.')
    print(json.dumps(summary, indent=2, default=str))
else:
    print('Set RUN_ALL = True and then use Run All to launch the overnight workflow.')

In [ ]:
summary_path = cfg.work_root / cfg.tag / 'morning_summary.json'
if summary_path.exists():
    print(summary_path)
    print(summary_path.read_text(encoding='utf-8')[:4000])
else:
    print('Morning summary not written yet:', summary_path)